In [2]:
import os
import sys
sys.path.append("kaggle/input/polymer_pipeline")

In [3]:
from data_preparation import get_data_paths, load_and_split_data
import model

In [4]:
os.environ['NEURIPS_DATA_PATH']     = 'kaggle/input/neurips-open-polymer-prediction-2025'
os.environ['EXTRA_DATA_BASE']       = 'kaggle/input/smiles-extra-data'
os.environ['TC_DATA_BASE']          = 'kaggle/input/tc-smiles'

In [5]:
paths = get_data_paths()
for k, v in paths.items():
    print(f"{k}: {v}")

train_csv: kaggle/input/neurips-open-polymer-prediction-2025/train.csv
test_csv: kaggle/input/neurips-open-polymer-prediction-2025/test.csv
sample_submission: kaggle/input/neurips-open-polymer-prediction-2025/sample_submission.csv
tc_data: kaggle/input/tc-smiles/Tc_SMILES.csv
tg_jcim_data: kaggle/input/smiles-extra-data/JCIM_sup_bigsmiles.csv
tg_excel_data: kaggle/input/smiles-extra-data/data_tg3.xlsx
density_data: kaggle/input/smiles-extra-data/data_dnst1.xlsx
supplement_dir: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement
ffv_data: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset4.csv
dataset1: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset1.csv
dataset2: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset2.csv
dataset3: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset3.csv


In [6]:
train_df, val_df, test_df = load_and_split_data(paths)
print("Loaded:", len(train_df), len(val_df), len(test_df))

👉 加载主训练数据
  原始训练样本数: 7973
  → 正在增强 Tc 数据，共 874 条
cross_smiles: 737
    填充已有样本 0 条，新增样本 129 条
  → 正在增强 Tg 数据，共 662 条
cross_smiles: 526
    填充已有样本 15 条，新增样本 136 条
  → 正在增强 Tg 数据，共 501 条
cross_smiles: 0
    填充已有样本 0 条，新增样本 499 条
  → 正在增强 Density 数据，共 787 条


[00:35:16] SMILES Parse Error: syntax error while parsing: *O[Si](*)([R])[R]
[00:35:16] SMILES Parse Error: Failed parsing SMILES '*O[Si](*)([R])[R]' for input: '*O[Si](*)([R])[R]'
[00:35:16] SMILES Parse Error: syntax error while parsing: *NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4
[00:35:16] SMILES Parse Error: Failed parsing SMILES '*NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4' for input: '*NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4'
[00:35:16] SMILES Parse Error: syntax error while parsing: O=C=N[R1]N=C=O.O[R2]O.O[R3]O
[00:35:16] SMILES Parse Error: Failed parsing SMILES 'O=C=N[R1]N=C=O.O[R2]O.O[R3]O' for input: 'O=C=N[R1]N=C=O.O[R2]O.O[R3]O'
[00:35:16] SMILES Parse Error: syntax error while parsing: *CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O
[00:35:16] SMILES Parse Error: Failed parsing SMILES '*CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O' for input: '*CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O'
[00:35:16] SMILES Parse 

cross_smiles: 254
    填充已有样本 110 条，新增样本 525 条
  → 正在增强 FFV 数据，共 862 条
cross_smiles: 43
    填充已有样本 43 条，新增样本 819 条
add dataset4: 10081
👉 划分 train / validation / test
  划分结果: train=8064, val=1008, test=1009
Loaded: 8064 1008 1009


In [7]:
import torch
import pandas as pd
from train_stage2 import optimize_stage2, train_final_stage2_model

/usr/local/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
train_df.head()

,id,SMILES,Tg,FFV,Tc,Density,Rg
0,2.026603e+09,*Oc1ccc(CC2(Cc3ccc(*)cc3)c3ccccc3-c3ccccc32)cc1,NaN,0.386736,NaN,NaN,NaN
1,1.189403e+09,*CC(O)COc1ccc(C(C)CC(C)(C)c2ccc(O*)cc2)cc1,NaN,0.354235,NaN,NaN,NaN
2,1.686537e+09,*C(=O)c1ccc2c(c1)C(=O)N(c1c(C)cc(C(c3cc(C)c(N4...,NaN,0.430396,NaN,NaN,NaN
3,2.632934e+08,*Oc1ccc2ccc(Oc3ccc(C(=Nc4ccc(N=C(c5ccccc5)c5cc...,NaN,0.381740,NaN,NaN,NaN
4,1.280165e+09,*Nc1ccc(-c2ccc(N*)c(OC)c2)cc1OC,NaN,0.334115,NaN,NaN,NaN


In [23]:
from rdkit import Chem
from rdkit.Chem import Descriptors

# 提供的非法 SMILES 列表
smiles_list = [
    r'*/C(=C(/*)c1ccc(C(C)(C)C)cc1)c1ccccc1',
    r'*/C(=C(/*)c1ccc(CCCC)cc1)c1ccccc1',
    r'*/C(=C(/*)c1ccc(Oc2ccccc2)cc1)c1ccccc1',
    r'*/C(=C(/*)c1ccc([Si](C(C)C)(C(C)C)C(C)C)cc1)c1ccccc1',
    r'*/C(=C(/*)c1ccc([Si](C)(C)C)cc1)c1ccccc1',
    r'*/C(=C(/*)c1cccc([Ge](C)(C)C)c1)c1ccccc1',
    r'*/C(=C(/*)c1cccc([Si](C)(C)C)c1)c1ccccc1',
    r'*/C(=C(/c1ccc(*)cc1)c1ccc(Oc2ccccc2)cc1)c1ccc(Oc2ccccc2)cc1',
    r'*/C(=C(\C#N)c1ccc(C(=O)OC2CCN(*)CC2)cc1)c1ccc(OC)cc1',
    r'*/C(=C(\[2H])C([2H])([2H])C(*)([2H])[2H])C([2H])([2H])[2H]',
    None  # 测试 None 也加入
]

print("SMILES 解析测试结果：\n")
for i, s in enumerate(smiles_list):
    if not s:
        print(f"{i}. SMILES = None → ❌ 无法解析（空值）")
        continue
    try:
        mol = Chem.MolFromSmiles(s)
        if mol:
            mw = Descriptors.MolWt(mol)
            print(f"{i}. SMILES = {s} → ✅ 成功解析，分子量: {mw:.2f}")
        else:
            print(f"{i}. SMILES = {s} → ❌ 无法解析（MolFromSmiles 返回 None）")
    except Exception as e:
        print(f"{i}. SMILES = {s} → ❌ 解析报错: {e}")

SMILES 解析测试结果：

0. SMILES = */C(=C(/*)c1ccc(C(C)(C)C)cc1)c1ccccc1 → ✅ 成功解析，分子量: 234.34
1. SMILES = */C(=C(/*)c1ccc(CCCC)cc1)c1ccccc1 → ✅ 成功解析，分子量: 234.34
2. SMILES = */C(=C(/*)c1ccc(Oc2ccccc2)cc1)c1ccccc1 → ✅ 成功解析，分子量: 270.33
3. SMILES = */C(=C(/*)c1ccc([Si](C(C)C)(C(C)C)C(C)C)cc1)c1ccccc1 → ✅ 成功解析，分子量: 334.58
4. SMILES = */C(=C(/*)c1ccc([Si](C)(C)C)cc1)c1ccccc1 → ✅ 成功解析，分子量: 250.42
5. SMILES = */C(=C(/*)c1cccc([Ge](C)(C)C)c1)c1ccccc1 → ✅ 成功解析，分子量: 294.94
6. SMILES = */C(=C(/*)c1cccc([Si](C)(C)C)c1)c1ccccc1 → ✅ 成功解析，分子量: 250.42
7. SMILES = */C(=C(/c1ccc(*)cc1)c1ccc(Oc2ccccc2)cc1)c1ccc(Oc2ccccc2)cc1 → ✅ 成功解析，分子量: 438.53
8. SMILES = */C(=C(\C#N)c1ccc(C(=O)OC2CCN(*)CC2)cc1)c1ccc(OC)cc1 → ✅ 成功解析，分子量: 360.41
9. SMILES = */C(=C(\[2H])C([2H])([2H])C(*)([2H])[2H])C([2H])([2H])[2H] → ✅ 成功解析，分子量: 76.17
10. SMILES = None → ❌ 无法解析（空值）


In [11]:
study = optimize_stage2(
    train_df=train_df,
    stage1_model_path="production_model.pth",
    study_name="stage2_graph_ssl",
    storage_uri="sqlite:///stage2_optuna.db",
    n_trials=50,
    tmp_dir="tmp_stage2",
    output_dir="stage2_artifacts"
)

📦 构建 PolymerDataset，样本数=8064
   成功转换为图数据: 8064 条
using device: cuda


[I 2025-08-02 00:37:48,758] A new study created in RDB with name: stage2_graph_ssl
[W 2025-08-02 00:38:23,423] Trial 0 failed with parameters: {'predictor_hidden_dim': 256, 'predictor_num_layers': 3, 'lr': 3.899697331966252e-05} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/miniconda3/lib/python3.12/site-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/root/kaggle-NeruIPS/kaggle/input/polymer_pipeline/train_stage2.py", line 152, in objective
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/kaggle-NeruIPS/kaggle/input/polymer_pipeline/train_stage2.py", line 85, in train_stage2_model
    pred = model(batch.x, batch.edge_index, batch.edge_attr, torch.ones(batch.edge_attr.size(0), device=device), batch.batch)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

KeyboardInterrupt: 

In [ ]:
# 加载最佳参数
params = torch.load("stage2_artifacts/stage2_best_params_trial0.pt")  # 根据实际 trial 号改名
params

GraphSSLModel(
  (encoder): WDMPNN()
  (predictor): GraphPredictor(
    (mlp): Sequential(
      (0): Linear(in_features=64, out_features=32, bias=True)
      (1): ReLU()
      (2): Linear(in_features=32, out_features=1, bias=True)
    )
  )
)
